# TPX3 -> HDF5 Conversion Diagnostics

Diagnoses the *direct* raw-to-h5 conversion path (`raw_to_h5.py`, wired into the "tpx to h5" Analysis tab) for one `.tpx3` file: decode it, sanity-check and plot the decoded pixels/TDC hits, convert it to h5, then verify the h5 file holds exactly what was decoded. No clustering, pulse-grouping, or timewalk correction happens anywhere in this path -- that's the cv4 pipeline (`raw_conversion.py`), not this one.

This hardware can mix several packet families in one raw file, across up to 4 chip streams:
- **Timepix3 pixel hits** (header nibble `0xb`)
- **PicoTDC hits** (header `0x6`, sub-nibble `0xC`) -- channel (0-63), edge (rising/falling), a `~3.05 ps` tick
- **FPGA TDC hits** (header `0x6`, sub-nibble `0xF`/`0xA`/`0xE`/`0xB`) -- two channels x two edges, a `~260.42 ps` tick
- occasionally other, still-unidentified packet types, which `raw_to_h5.py` deliberately skips rather than guess at

`raw_to_h5.decode_raw_file` reads every packet by its literal header nibble rather than guessing a pixel/TDC header pair by frequency (as `tpx_processing.decode_tpx3`'s `detect_headers` does) -- that guess has been observed to mislabel pixel and TDC packets on files where a third, unidentified packet family dominates.

If the file you point at in step 1 doesn't exist, this notebook builds a small synthetic `.tpx3` file instead (one pixel hit, one PicoTDC pulse, one FPGA TDC pulse), so the whole notebook runs as a self-test with no real data on hand.

Run from the project root (same folder as `raw_to_h5.py`) so the imports below resolve.

In [ ]:
import pathlib
import time

import h5py
import numpy as np
import matplotlib.pyplot as plt

import raw_to_h5

%matplotlib inline

## 1. Input file

Point `tpx3_path` at a raw `.tpx3` file (a Monitored Acquisition / Collection parameters raw folder, or one position's raw file from a Parameter Sweep). If it isn't found, the next cell builds a small synthetic file instead.

In [ ]:
TPX3_MAGIC = (0x33585054).to_bytes(4, "little")  # 'TPX3' chunk magic, Table 6.1


def _chunk(chip, words):
    """One chunk: header (magic + chip index + payload length) + packets."""
    payload = np.array(words, dtype=np.uint64).tobytes()
    header = TPX3_MAGIC + bytes([chip & 0xFF, 0]) + len(payload).to_bytes(2, "little")
    return header + payload


def _encode_pixel_word(x, y, tot_counts, toa_ns):
    """Inverse of decode_raw_file's pixel unpacking -- header 0xb directly, no shift."""
    toa_count = int(round(toa_ns / raw_to_h5.PIXEL_RES_NS))
    ftoa = (-toa_count) % 16
    q = (toa_count + ftoa) // 16
    coarse = (q // 16384) & 0xFFFF
    toa = q % 16384
    pix = (x % 2) * 4 + (y % 4)
    pix_addr = ((x // 2) << 9) | ((y // 4) << 3) | pix
    word = (
        (0xB << 60)
        | (pix_addr << 44)
        | (toa << 30)
        | ((tot_counts & 0x3FF) << 20)
        | ((ftoa & 0xF) << 16)
        | (coarse & 0xFFFF)
    )
    return word


def _encode_picotdc_word(channel, edge, ts_count, valid=0):
    """Header 0x6, sub 0xC -- channel/edge/valid/spidr/timestamp_lower layout.
    Inverse of timestamp = (spidr_time << 26) | timestamp_lower: spidr_time is
    the top 21 bits of the combined count, timestamp_lower the bottom 26."""
    spidr = (ts_count >> 26) & 0x1FFFFF
    ts_lower = ts_count & 0x3FFFFFF
    chan_hi = (channel >> 4) & 0x3
    chan_lo = channel & 0xF
    word = (
        (0x6 << 60)
        | (0xC << 56)
        | (chan_hi << 53)
        | ((valid & 0x1) << 52)
        | (chan_lo << 48)
        | ((edge & 0x1) << 47)
        | (ts_lower << 21)
        | spidr
    )
    return word


def _encode_fpga_tdc_word(sub_value, combined_count):
    """Header 0x6, sub in {0xF, 0xA, 0xE, 0xB} -- coarse*12 + (fine-1) combined count."""
    coarse, fine_m1 = divmod(combined_count, 12)
    fine = fine_m1 + 1  # 1..12; 0 means 'error' and is dropped by decode_raw_file
    word = (0x6 << 60) | (sub_value << 56) | ((coarse & ((1 << 35) - 1)) << 9) | ((fine & 0xF) << 5)
    return word


def make_synthetic_tpx3(path, n_pulses=8, pulse_period_ns=1_000_000.0, seed=0):
    """A small raw file, chip stream 0, with known pixel hits, PicoTDC rising/falling
    edges (channel 28, ~30 ns wide pulses) and FPGA TDC rising/falling edges
    (channel 1), all written in time order like a real acquisition."""
    rng = np.random.default_rng(seed)
    timed_words = []
    t = 1000.0
    pico_width_ns = 30.0
    fpga_width_ns = 60.0
    for _ in range(n_pulses):
        timed_words.append((t, _encode_pixel_word(
            int(rng.integers(0, 256)), int(rng.integers(0, 256)),
            int(rng.integers(0, 1024)), t,
        )))
        pico_rise = int(round(t / raw_to_h5.PICOTDC_NS_PER_COUNT))
        pico_fall = int(round((t + pico_width_ns) / raw_to_h5.PICOTDC_NS_PER_COUNT))
        timed_words.append((t, _encode_picotdc_word(28, 1, pico_rise)))
        timed_words.append((t + pico_width_ns, _encode_picotdc_word(28, 0, pico_fall)))
        fpga_rise = int(round(t / raw_to_h5.FPGA_TDC_NS_PER_COUNT))
        fpga_fall = int(round((t + fpga_width_ns) / raw_to_h5.FPGA_TDC_NS_PER_COUNT))
        timed_words.append((t, _encode_fpga_tdc_word(0xF, fpga_rise)))
        timed_words.append((t + fpga_width_ns, _encode_fpga_tdc_word(0xA, fpga_fall)))
        t += pulse_period_ns + float(rng.uniform(-10.0, 10.0))

    timed_words.sort(key=lambda tw: tw[0])
    words = [w for _, w in timed_words]
    pathlib.Path(path).write_bytes(_chunk(0, words))

In [ ]:
tpx3_path = pathlib.Path(r"C:\DATA\SB Test\fJgN_000000.tpx3")  # <-- point this at a real raw file
is_synthetic = not tpx3_path.is_file()

if is_synthetic:
    tpx3_path = pathlib.Path("tpx_to_h5_diagnostics_synthetic.tpx3")
    make_synthetic_tpx3(tpx3_path)
    print(f"Input file not found -- using a synthetic self-test file instead: {tpx3_path}")
else:
    print(f"Using: {tpx3_path}")

## 2. Decode

`raw_to_h5.decode_raw_file` -- the same function the "tpx to h5" Analysis tab calls per raw file. No clustering or pulse-grouping yet, just every pixel hit and every TDC hit, decoded by its packet's own header nibble.

In [ ]:
t0 = time.perf_counter()
data = raw_to_h5.decode_raw_file(str(tpx3_path))
decode_time = time.perf_counter() - t0

print(f"Decoded in {decode_time:.3f} s")
print(f"Pixel hits: {len(data['pixel_x'])}")
print(f"TDC hits:   {len(data['tdc_time_ns'])}")

if len(data["tdc_type"]):
    types, type_counts = np.unique(data["tdc_type"], return_counts=True)
    print("  by tdc_type:", {f"{int(t):#x}": int(c) for t, c in zip(types, type_counts)})
    pico = data["tdc_type"] == 0xC
    if pico.any():
        chans, chan_counts = np.unique(data["tdc_channel"][pico], return_counts=True)
        print("  PicoTDC channels:", dict(zip(chans.tolist(), chan_counts.tolist())))

## 2a. Example raw packets

What `decode_raw_file` actually reads before it becomes pixels/TDC hits: every packet is a 64-bit word, `header = word >> 60` picks the packet family, and (for TDC packets) the next nibble down picks PicoTDC vs. FPGA TDC. No `-2**62` shift here -- unlike `tpx_processing.decode_tpx3`, this reads the literal on-disk nibble.

In [ ]:
EXAMPLES_PER_HEADER = 3
header_counts = np.zeros(16, dtype=np.int64)
example_packets = {}

for chip, words in raw_to_h5.iter_chip_tagged_chunks(str(tpx3_path)):
    headers = (words >> 60).astype(np.int64)
    header_counts += np.bincount(headers, minlength=16)
    for h in np.unique(headers):
        bucket = example_packets.setdefault(int(h), [])
        if len(bucket) < EXAMPLES_PER_HEADER:
            need = EXAMPLES_PER_HEADER - len(bucket)
            bucket.extend(words[headers == h][:need].tolist())

print("Packet header counts (top 4 bits of every word):")
for h in range(16):
    if not header_counts[h]:
        continue
    tag = "pixel" if h == 0xB else ("TDC (PicoTDC/FPGA)" if h == 0x6 else "not decoded")
    print(f"  {h:#x}: {header_counts[h]:>10}  ({tag})")

### Pixel packets (header `0xb`)

`pix_addr` (bits 44-59) gives (x, y); `toa` (14 b) + `coarse` (16 b) + `ftoa` (4 b) give ToA at `25/16 ns` per count; `tot` (10 b) gives ToT at `25 ns` per count.

In [ ]:
pixel_examples = example_packets.get(0xB, [])
if not pixel_examples:
    print("No pixel packets found.")

for packet in pixel_examples:
    encoded = packet >> 44
    dcol = (encoded >> 8) & 0x7F
    spix = (encoded >> 1) & 0xFF
    pix = encoded & 0x7
    x = dcol + pix // 4
    y = spix + (pix & 0x3)
    ftoa = (packet >> 16) & 0xF
    toa = (packet >> 30) & 0x3FFF
    coarse = packet & 0xFFFF
    toa_count = (((coarse << 14) + toa) << 4) - ftoa
    tot = (packet >> 20) & 0x3FF
    print(f"raw word: {packet:#018x}")
    print(f"  x={x}  y={y}  tot={tot * 25.0:.0f} ns  toa={toa_count * raw_to_h5.PIXEL_RES_NS:.3f} ns\n")

### PicoTDC packets (header `0x6`, sub `0xC`)

`channel` (0-63), `valid` (0 = valid hit), `edge` (1 = rising), and a combined `(spidr_time << 26) | timestamp_lower` counter at `25/8192 ns ~= 3.05 ps` per count.

In [ ]:
picotdc_examples = [p for p in example_packets.get(0x6, []) if (p >> 56) & 0xF == 0xC]
if not picotdc_examples:
    print("No PicoTDC packets found.")

for packet in picotdc_examples:
    channel = (((packet >> 53) & 0x3) << 4) | ((packet >> 48) & 0xF)
    valid = (packet >> 52) & 0x1
    edge = (packet >> 47) & 0x1
    ts_lower = (packet >> 21) & 0x3FFFFFF
    spidr = packet & 0x1FFFFF
    ts = (spidr << 26) | ts_lower
    print(f"raw word: {packet:#018x}")
    print(
        f"  channel={channel}  valid={'skip' if valid else 'valid'}  "
        f"edge={'rising' if edge else 'falling'}  -> time={ts * raw_to_h5.PICOTDC_NS_PER_COUNT:.3f} ns\n"
    )

### FPGA TDC packets (header `0x6`, sub `0xf`/`0xa`/`0xe`/`0xb`)

Two channels (`0xf`/`0xa` = channel 1 rising/falling, `0xe`/`0xb` = channel 2 rising/falling), a `coarse * 12 + (fine - 1)` combined counter at `3.125/12 ns ~= 260.42 ps` per count.

In [ ]:
_FPGA_LABELS = {0xF: "ch1 rising", 0xA: "ch1 falling", 0xE: "ch2 rising", 0xB: "ch2 falling"}
fpga_examples = [p for p in example_packets.get(0x6, []) if (p >> 56) & 0xF in _FPGA_LABELS]
if not fpga_examples:
    print("No FPGA TDC packets found.")

for packet in fpga_examples:
    sub = (packet >> 56) & 0xF
    fine = (packet >> 5) & 0xF
    coarse = (packet >> 9) & ((1 << 35) - 1)
    combined = coarse * 12 + (fine - 1) if fine else None
    print(f"raw word: {packet:#018x}")
    print(f"  sub={sub:#x} ({_FPGA_LABELS[sub]})  fine={fine}" + (
        f"  -> time={combined * raw_to_h5.FPGA_TDC_NS_PER_COUNT:.3f} ns\n" if fine else "  (0 = error, dropped)\n"
    ))

### Other headers (not decoded by `raw_to_h5.py`)

Anything besides `0xb` and `0x6` is skipped -- a non-trivial count here (this hardware has at least one still-unidentified packet family) is worth a look before trusting a file's pixel/TDC counts are the whole story.

In [ ]:
other_headers = [h for h in example_packets if h not in (0xB, 0x6)]
if not other_headers:
    print("No other packet headers found.")
for h in sorted(other_headers):
    print(f"header {h:#x}  ({header_counts[h]} packet(s)):")
    for packet in example_packets[h]:
        print(f"  raw word: {packet:#018x}")

## 3. Sanity checks

In [ ]:
xs = data["pixel_x"].astype(np.int64)
ys = data["pixel_y"].astype(np.int64)
tots = data["pixel_tot"]
toas = data["pixel_toa_ns"]


def check(label, condition):
    print(f"[{'OK' if condition else 'WARN'}] {label}")


if len(xs):
    check(f"x in [0, 255]: min={xs.min()}, max={xs.max()}", xs.min() >= 0 and xs.max() <= 255)
    check(f"y in [0, 255]: min={ys.min()}, max={ys.max()}", ys.min() >= 0 and ys.max() <= 255)
    out_of_order = int(np.sum(np.diff(toas) < 0))
    check(f"pixel toa mostly increasing ({out_of_order} out-of-order step(s))", out_of_order < 0.01 * len(toas))
else:
    print("No pixels decoded.")

if len(data["tdc_chip"]):
    pchips, pchip_counts = np.unique(data["pixel_chip"], return_counts=True) if len(data["pixel_chip"]) else ([], [])
    tchips, tchip_counts = np.unique(data["tdc_chip"], return_counts=True)
    print(f"Pixel hits by chip: {dict(zip(pchips.tolist(), pchip_counts.tolist()))}")
    print(f"TDC hits by chip:   {dict(zip(tchips.tolist(), tchip_counts.tolist()))}")

## 4. Diagnostic plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

ax = axes[0, 0]
if len(xs):
    h = ax.hist2d(xs, ys, bins=256, range=[[0, 256], [0, 256]], cmap="viridis")
    fig.colorbar(h[3], ax=ax, label="counts")
ax.set_title("Pixel hit map")
ax.set_xlabel("x")
ax.set_ylabel("y")

ax = axes[0, 1]
if len(tots):
    ax.hist(tots, bins=64)
ax.set_title("ToT")
ax.set_xlabel("tot (ns)")

ax = axes[0, 2]
if len(toas):
    ax.hist((toas - toas.min()) / 1e6, bins=200)
ax.set_title("Pixel rate over the file")
ax.set_xlabel("time since first hit (ms)")
ax.set_ylabel("hits / bin")

ax = axes[1, 0]
if len(data["tdc_type"]):
    groups, group_counts = np.unique(
        list(zip(data["tdc_type"].tolist(), data["tdc_channel"].tolist())), axis=0, return_counts=True
    )
    labels = [f"{int(t):#x}/ch{int(c)}" if c >= 0 else f"{int(t):#x}" for t, c in groups]
    ax.bar(labels, group_counts)
    ax.tick_params(axis="x", rotation=45)
ax.set_title("TDC hits by type/channel")

ax = axes[1, 1]
tdc_time = data["tdc_time_ns"]
if len(tdc_time):
    for t_type in np.unique(data["tdc_type"]):
        m = data["tdc_type"] == t_type
        ax.hist((tdc_time[m] - tdc_time.min()) / 1e6, bins=100, alpha=0.6, label=f"{int(t_type):#x}")
    ax.legend(fontsize=8)
ax.set_title("TDC hit rate over the file")
ax.set_xlabel("time since first TDC hit (ms)")

ax = axes[1, 2]
pico = data["tdc_type"] == 0xC
if pico.any():
    first_channel = int(np.unique(data["tdc_channel"][pico])[0])
    m = pico & (data["tdc_channel"] == first_channel)
    chip_m = data["tdc_chip"][m]
    edge_m = data["tdc_edge"][m]
    time_m = tdc_time[m]
    widths = []
    for c in np.unique(chip_m):
        cm = chip_m == c
        order = np.argsort(time_m[cm])
        e = edge_m[cm][order]
        t = time_m[cm][order]
        i = 0
        while i < len(e) - 1:
            if e[i] != e[i + 1]:
                widths.append(abs(t[i + 1] - t[i]))
                i += 2
            else:
                i += 1
    if widths:
        ax.hist(widths, bins=30)
        ax.set_title(f"PicoTDC ch{first_channel} pulse width")
        ax.set_xlabel("width (ns)")
else:
    ax.set_title("No PicoTDC channel found")

fig.tight_layout()

## 5. Convert directly to h5

`raw_to_h5.convert_file` -- the same function the "tpx to h5" Analysis tab calls per raw file.

In [ ]:
h5_path = tpx3_path.with_suffix(".h5")
t0 = time.perf_counter()
counts = raw_to_h5.convert_file(tpx3_path, h5_path)
convert_time = time.perf_counter() - t0
print(f"Converted in {convert_time:.3f} s -> {h5_path}")
print(counts)

## 6. Verify the h5 matches the decode

Reopens the h5 file and checks its datasets are exactly the decoded pixels/TDC hits -- same values, same order, nothing dropped or reshuffled.

In [ ]:
expected = {
    "x": data["pixel_x"], "y": data["pixel_y"], "toa": data["pixel_toa_ns"],
    "tot": data["pixel_tot"], "pixel_chip": data["pixel_chip"],
    "tdc_time": data["tdc_time_ns"], "tdc_type": data["tdc_type"],
    "tdc_channel": data["tdc_channel"], "tdc_edge": data["tdc_edge"], "tdc_chip": data["tdc_chip"],
}

with h5py.File(h5_path, "r") as f:
    all_ok = True
    for name, expected_values in expected.items():
        actual = f[name][:]
        ok = np.array_equal(actual, expected_values)
        all_ok &= ok
        print(f"[{'OK' if ok else 'FAIL'}] {name}: {len(actual)} value(s)")
    print()
    print("Attributes:")
    for key, value in f.attrs.items():
        print(f"  {key}: {value}")

print()
print("PASS -- h5 matches the decode exactly" if all_ok else "FAIL -- see mismatches above")

## 7. Cleanup

Only removes files this notebook created for the synthetic self-test; a real input file (and its h5) are left alone.

In [ ]:
if is_synthetic:
    tpx3_path.unlink(missing_ok=True)
    h5_path.unlink(missing_ok=True)
    print("Removed the synthetic self-test files.")
else:
    print(f"Keeping {tpx3_path} and {h5_path}.")